# Import Libraries

In [2]:
import sys
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from statistics import mean
from rdkit.Chem import Descriptors, rdMolDescriptors, PandasTools, AllChem
from scipy.stats import loguniform, randint, uniform
from boruta import BorutaPy
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score, confusion_matrix,matthews_corrcoef
from sklearn.metrics import make_scorer, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, f1_score, roc_curve, precision_score, cohen_kappa_score, accuracy_score, r2_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, KFold
from sklearn.model_selection import cross_validate
from sklearn.utils import resample
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Load your descriptor file here

In [3]:
data = pd.read_csv(r"Dataset_external_descriptors_NLRP.csv")

In [4]:
data

,Code,Smiles,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,IS00001,O=C(c1ccccc1)c1ccccc1.CC1(C)CC2OC3C(OCc4ccccc4...,11.846296,11.846296,0.030966,-0.144561,0.186839,24.837209,570.729,532.425,...,0,0,0,0,0,0,0,0,0,0
1,IS00002,CCOC(=O)C1=C(C)NC(=O)NC1c1ccc(cc1)N(C)C,12.194467,12.194467,0.287547,-0.510174,0.834057,17.636364,303.362,282.194,...,0,0,0,0,0,0,0,0,0,1
2,IS00003,CC1Oc2cc(O)ccc2C(=O)C1C,11.776612,11.776612,0.082176,-0.127963,0.683360,25.428571,192.214,180.118,...,0,0,0,0,0,0,0,0,0,0
3,IS00004,COc1ccc2C(=O)CC(C)(C)Oc2c1,11.779947,11.779947,0.128614,-0.420190,0.707556,17.933333,206.241,192.129,...,0,0,0,0,0,0,0,0,0,0
4,IS00005,Oc1ccc2C(=O)CC3(CCCC3)Oc2c1,11.996357,11.996357,0.141806,-0.279884,0.727797,21.875000,218.252,204.140,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2169,IS002501,O=C(N1CCCCC1)c1ccccc1,11.925648,11.925648,0.185463,0.185463,0.663533,16.714286,189.258,174.138,...,0,0,0,0,0,0,0,0,0,0
2170,IS002502,Cc1ccc(N2CCCC2)c(C)c1,2.494815,2.494815,1.239981,1.239981,0.634054,16.615385,175.275,158.139,...,0,0,0,0,0,0,0,0,0,0
2171,IS002503,O=C(CCc1ccccc1)C#Cc1ccccc1,11.616080,11.616080,0.011383,-0.011383,0.745040,9.333333,234.298,220.186,...,0,0,0,0,0,0,0,0,0,0
2172,IS002504,OC(=O)CCCNC(=O)c1ccccc1,11.430811,11.430811,0.078442,-0.843555,0.714609,9.600000,207.229,194.125,...,0,0,0,0,0,0,0,0,0,0


In [7]:
eX_data =data.iloc[:, 2:]

In [8]:
eX_data

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,11.846296,11.846296,0.030966,-0.144561,0.186839,24.837209,570.729,532.425,570.277010,218,...,0,0,0,0,0,0,0,0,0,0
1,12.194467,12.194467,0.287547,-0.510174,0.834057,17.636364,303.362,282.194,303.158292,118,...,0,0,0,0,0,0,0,0,0,1
2,11.776612,11.776612,0.082176,-0.127963,0.683360,25.428571,192.214,180.118,192.078644,74,...,0,0,0,0,0,0,0,0,0,0
3,11.779947,11.779947,0.128614,-0.420190,0.707556,17.933333,206.241,192.129,206.094294,80,...,0,0,0,0,0,0,0,0,0,0
4,11.996357,11.996357,0.141806,-0.279884,0.727797,21.875000,218.252,204.140,218.094294,84,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2169,11.925648,11.925648,0.185463,0.185463,0.663533,16.714286,189.258,174.138,189.115364,74,...,0,0,0,0,0,0,0,0,0,0
2170,2.494815,2.494815,1.239981,1.239981,0.634054,16.615385,175.275,158.139,175.136100,70,...,0,0,0,0,0,0,0,0,0,0
2171,11.616080,11.616080,0.011383,-0.011383,0.745040,9.333333,234.298,220.186,234.104465,88,...,0,0,0,0,0,0,0,0,0,0
2172,11.430811,11.430811,0.078442,-0.843555,0.714609,9.600000,207.229,194.125,207.089543,80,...,0,0,0,0,0,0,0,0,0,0


# Normilization of data

In [9]:
with open("standard_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
data_scaled = scaler.transform(eX_data)

In [10]:
list2=eX_data.columns
external1= pd.DataFrame(data_scaled,columns=list2)

# Selected fatures

In [11]:
with open("selected_features.pkl", "rb") as f:
    selected_features = pickle.load(f)
# Apply to new data
selected = external1[selected_features]

In [12]:
selected

,MinEStateIndex,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,BCUT2D_MWHI,BCUT2D_CHGLO,BCUT2D_LOGPLOW,BCUT2D_MRHI,Chi0,...,SMR_VSA5,SMR_VSA7,SMR_VSA9,SlogP_VSA6,TPSA,EState_VSA8,VSA_EState1,VSA_EState2,VSA_EState6,MolLogP
0,0.208100,1.616151,1.543305,1.617996,2.080000,-0.995539,-1.461693,-0.468258,-0.528947,2.178307,...,0.770703,4.475454,2.045400,3.128108,-1.193512,2.499912,0.293639,-0.432622,3.666557,3.477051
1,0.109960,-0.735744,-0.774100,-0.736343,-0.586667,-0.998836,-1.190638,-0.677587,-0.598879,-0.555205,...,-0.560866,-0.569146,0.000000,0.073860,-0.411618,-0.383873,-0.235501,0.839748,0.104456,-0.847526
2,0.212556,-1.713458,-1.719432,-1.715381,-1.760000,-1.001393,-1.111372,0.020444,-0.565891,-1.778079,...,-0.558996,-1.426730,1.955620,-0.590295,-1.140098,-1.056753,-0.220245,-0.438922,-0.161218,-0.810880
3,0.134114,-1.590070,-1.608197,-1.591849,-1.600000,-1.000483,-1.508267,-0.121634,-0.571392,-1.622845,...,-0.380888,-1.426730,1.955620,-0.590295,-1.472048,-1.056753,-0.022726,-0.438621,-0.079277,-0.471660
4,0.171776,-1.484415,-1.496962,-1.486083,-1.493333,-1.000496,-1.817138,-0.321183,-0.570773,-1.598048,...,-0.024672,-1.426730,1.955620,-0.590295,-1.140098,-1.056753,-0.204012,-0.419056,-0.144910,-0.296246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2169,0.296688,-1.739461,-1.774813,-1.741499,-1.760000,-1.023004,-1.008460,-0.103051,-0.603120,-1.878108,...,-0.579732,-0.826561,0.000000,-0.125495,-1.931347,-1.056753,-0.425397,-0.248510,0.418640,-0.567485
2170,0.579748,-1.862462,-1.922980,-1.864709,-1.866667,-1.087371,-0.353644,0.885373,-0.802075,-1.989289,...,-0.356216,-1.151523,0.000000,-0.590295,-2.446473,1.281177,-0.425397,-1.278044,0.090769,-0.119133
2171,0.243849,-1.343266,-1.348359,-1.344972,-1.386667,-1.023988,2.052849,2.159436,-0.593548,-1.300014,...,-0.772976,0.949070,2.013757,1.036506,-2.029121,-0.307477,-0.425397,-0.453435,1.592714,0.136067
2172,0.020472,-1.581379,-1.589712,-1.583077,-1.600000,-1.009493,1.356833,0.523034,-0.603848,-1.613858,...,-0.772976,-0.826561,0.000000,-0.125495,-0.540475,-0.720313,-0.425397,0.452062,0.335986,-1.350096


In [14]:
import pandas as pd
import pickle

# Load saved models
with open("dt_model.pkl", "rb") as f:
    mod_dt = pickle.load(f)

with open("adbt_model.pkl", "rb") as f:
    mod_adbt = pickle.load(f)

# External feature set
df_predict_features = selected

# Predictions
df_predict_clf = pd.DataFrame()
df_predict_clf['Class_pred_DecisionTree'] = mod_dt.predict(df_predict_features)
df_predict_clf['Class_pred_AdaBoost'] = mod_adbt.predict(df_predict_features)

# Metadata
t1 = pd.DataFrame(data['Code'].values, columns=['Code'])
t2 = pd.DataFrame(data['Smiles'].values, columns=['Smiles'])

# Final output
result = pd.concat([t1, t2, df_predict_clf], axis=1)

,Code,Smiles,Class_pred_DecisionTree,Class_pred_AdaBoost
0,IS00001,O=C(c1ccccc1)c1ccccc1.CC1(C)CC2OC3C(OCc4ccccc4...,0,0
1,IS00002,CCOC(=O)C1=C(C)NC(=O)NC1c1ccc(cc1)N(C)C,0,0
2,IS00003,CC1Oc2cc(O)ccc2C(=O)C1C,0,0
3,IS00004,COc1ccc2C(=O)CC(C)(C)Oc2c1,0,0
4,IS00005,Oc1ccc2C(=O)CC3(CCCC3)Oc2c1,0,0
...,...,...,...,...
2169,IS002501,O=C(N1CCCCC1)c1ccccc1,0,0
2170,IS002502,Cc1ccc(N2CCCC2)c(C)c1,0,0
2171,IS002503,O=C(CCc1ccccc1)C#Cc1ccccc1,0,0
2172,IS002504,OC(=O)CCCNC(=O)c1ccccc1,0,0


In [18]:
result.to_csv('predictions_external.csv')